# 02613 — Python and High-Performance Computing
## All Exam & Re-exam Questions with Full Solutions

**Covers:** Exam 2024 · Re-exam 2024 · F25 question set

Each question includes:
- The original question
- The correct answer
- A step-by-step explanation of **how to solve it**
- Runnable code where relevant

---

# EXAM 2024 (28.05.2024)

## Q1 — LSF Job Script: Fix the resource specifications

**Question:** The following job script has wrong resource specs. The program runs best with 8 cores, uses 16 GB memory, and takes at most 15 minutes. Fix it.

```bash
#!/bin/bash
#BSUB -J proc
#BSUB -q hpc
#BSUB -W 00:05          ← wrong time
#BSUB -R "rusage[mem=4GB]"  ← wrong memory
#BSUB -n 1              ← wrong cores
#BSUB -o proc_%J.out
#BSUB -e proc_%J.err
python process.py allthe.data
```

---

### ✅ Answer
```bash
#BSUB -W 00:15
#BSUB -n 8
#BSUB -R "rusage[mem=2GB]"
#BSUB -R "span[hosts=1]"
```

### 🔍 How to solve it

1. **Wall time**: program takes at most 15 minutes → `00:15`
2. **Cores**: program runs best with 8 cores → `-n 8`
3. **Memory**: LSF memory is specified **per core**, not total.
   - Total needed = 16 GB, cores = 8
   - Per core = 16 / 8 = **2 GB** → `rusage[mem=2GB]`
4. **span[hosts=1]**: when using multiple cores for shared-memory parallelism, all cores must be on the same node. Always add this when `-n > 1`.

## Q2 — Amdahl's Law: Estimate parallel fraction from a speedup plot

**Question:** A speedup plot shows the curve plateauing at a speedup of 5. What is the parallel fraction?

Options: (a) 0.2  (b) 0.5  (c) 0.8  (d) 0.9

---

### ✅ Answer: (c) 0.8

### 🔍 How to solve it

The speedup plateau = theoretical maximum = $\frac{1}{1-F}$

$$\text{max speedup} = 5 \implies 1 - F = \frac{1}{5} \implies F = 0.8$$

**General formula:**
$$F = 1 - \frac{1}{\text{plateau speedup}}$$

In [1]:
# Verify: with F=0.8, what is the max speedup?
F = 0.8
max_speedup = 1 / (1 - F)
print(f"F = {F} → max speedup = {max_speedup}")  # 5.0

# Read F from a plateau value
plateau = 5
F_estimated = 1 - 1/plateau
print(f"Plateau = {plateau} → F = {F_estimated}")

F = 0.8 → max speedup = 5.000000000000001
Plateau = 5 → F = 0.8


## Q3 — Amdahl's Law: Should they pursue parallelisation?

**Question:** Management wants speedup ≥ 4. The best machine has 8 cores. With F = 0.8, should they pursue parallelisation?

---

### ✅ Answer: No

### 🔍 How to solve it

Apply Amdahl's law with $F = 0.8$, $p = 8$:

$$S(8) = \frac{1}{(1-F) + F/p} = \frac{1}{0.2 + 0.8/8} = \frac{1}{0.2 + 0.1} = \frac{1}{0.3} = 3.33$$

$3.33 < 4$ → target not achievable → **do not pursue parallelisation**.

In [2]:
def amdahl(F, p):
    return 1 / ((1 - F) + F / p)

F, p, target = 0.8, 8, 4
S = amdahl(F, p)
print(f"S({p}) = {S:.3f}")
print(f"Meets target of {target}? {'Yes' if S >= target else 'No'}")

S(8) = 3.333
Meets target of 4? No


## Q4 — Parallelisation Approach: Threading or Processing?

**Question:** The following function is applied to 100 million numbers. Which parallelisation approach?

```python
def process_number(n):
    s = 0
    for i in range(n):
        s += i
    return s
```

Options: (a) Multi-threading  (b) Multi-processing  (c) Does not matter

---

### ✅ Answer: (b) Multi-processing

### 🔍 How to solve it

The function is a **pure Python loop** (no NumPy, no Numba). The Python GIL prevents true parallel execution of Python bytecode in threads.

**Decision rule:**
- Pure Python CPU work → **multiprocessing** (bypasses GIL)
- NumPy / Numba with `nogil=True` → **threading** is fine (GIL released)
- I/O-bound work → threading is fine

## Q5 — Scheduling: Static or Dynamic?

**Question:** Each number in `[1, 2, ..., 100_000_000]` is processed by `process_number(n)`. Should they use static or dynamic scheduling?

---

### ✅ Answer: Dynamic scheduling

### 🔍 How to solve it

`process_number(n)` loops `n` times. For n=1 it's nearly instant; for n=100,000,000 it takes a very long time. **Task duration varies enormously** with the input.

- **Static**: divide 100M numbers equally upfront → one worker gets all the large numbers and takes much longer → other workers idle.
- **Dynamic**: workers pick the next number as they finish → work distributes evenly.

**Rule:** Use dynamic when `StdDev(task time)` is large relative to `Mean(task time)`.

## Q6 — Parallel Reduction: Why does abssum fail?

**Question:** Why can't this function be used in a parallel reduction? What should they do instead?

```python
def abssum(x, y):
    return abs(x + y)
```

---

### ✅ Answer: The function is not associative

### 🔍 How to solve it

Parallel reduction requires **associativity**: `f(f(a,b), c) == f(a, f(b,c))` for all inputs.

Counter-example with values 1, 2, -3:
- Left-to-right: `abs(abs(1+2) + (-3)) = abs(3 - 3) = 0`
- Right-to-left: `abs(1 + abs(2+(-3))) = abs(1 + 1) = 2`
- `0 ≠ 2` → **not associative** → cannot use in reduction.

**Fix:** Do a normal parallel sum, then take `abs` at the very end:
```python
result = abs(sum(values))  # sum is associative ✅
```

In [3]:
# Demonstrate non-associativity of abs(x+y)
def abssum(x, y):
    return abs(x + y)

a, b, c = 1, 2, -3

left_assoc = abssum(abssum(a, b), c)   # (|1+2|) + (-3)
right_assoc = abssum(a, abssum(b, c))  # 1 + (|2+(-3)|)

print(f"abssum(abssum({a},{b}), {c}) = {left_assoc}")
print(f"abssum({a}, abssum({b},{c})) = {right_assoc}")
print(f"Associative? {left_assoc == right_assoc}")

# Correct approach
values = [a, b, c]
print(f"\nCorrect: abs(sum({values})) = {abs(sum(values))}")

abssum(abssum(1,2), -3) = 0
abssum(1, abssum(2,-3)) = 2
Associative? False

Correct: abs(sum([1, 2, -3])) = 0


## Q7 — NumPy Broadcasting: Subtract mean image

**Question:** `images` has shape `(N, H, W, 3)`. `mim` (mean image) has shape `(H, W)`. Which operation subtracts `mim` from each channel of every image?

Options:
- (a) `images - mim`
- (b) `images - mim[:, :, None]`
- (c) `images - mim[None]`

---

### ✅ Answer: (b) `images - mim[:, :, None]`

### 🔍 How to solve it

Work out the shapes after indexing, then check broadcast compatibility:

```
images shape:          (N, H, W, 3)

(a) mim:               (H, W)          → pad left → (1, 1, H, W)
    broadcast result:  (N, 1, H, W) — wrong, misaligned with (N,H,W,3)

(b) mim[:,:,None]:     (H, W, 1)       → pad left → (1, H, W, 1)
    broadcast result:  (N, H, W, 3) ✅  — 1 broadcasts over 3 channels

(c) mim[None]:         (1, H, W)       → pad left → (1, 1, H, W)
    broadcast result:  same issue as (a)
```

**Rule:** Add a `None` axis at the end to broadcast over a trailing axis.

In [4]:
import numpy as np

N, H, W = 4, 8, 8
images = np.random.rand(N, H, W, 3)
mim = np.random.rand(H, W)

# (b) correct
result = images - mim[:, :, None]
print("mim[:,:,None] shape:", mim[:, :, None].shape)  # (H, W, 1)
print("result shape:", result.shape)                    # (N, H, W, 3)

# (a) would fail or give wrong shape
try:
    r = images - mim
    print("(a) shape:", r.shape)  # may fail
except ValueError as e:
    print("(a) fails:", e)

mim[:,:,None] shape: (8, 8, 1)
result shape: (4, 8, 8, 3)
(a) fails: operands could not be broadcast together with shapes (4,8,8,3) (8,8) 


## Q8 — Cache Efficiency: Reorder loops using strides

**Question:** The `images` array has strides `(600, 40, 8, 200)` for axes `(i, j, k, l)`. How should the loops be reordered?

```python
for i in range(images.shape[0]):
    for j in range(images.shape[1]):
        for k in range(images.shape[2]):
            for l in range(images.shape[3]):
                x = images[i, j, k, l]
```

---

### ✅ Answer: Innermost loop should be axis `k` (stride 8). Order from outer to inner: `i, l, j, k`

### 🔍 How to solve it

1. List the strides for each axis:

| Axis | Stride |
|------|--------|
| i    | 600    |
| j    | 40     |
| k    | **8** ← smallest |
| l    | 200    |

2. **Innermost loop = smallest stride** (most cache-friendly — each step moves only 8 bytes, staying in the same cache line).

3. Sort from largest stride (outermost) to smallest (innermost):
   - i (600) → l (200) → j (40) → k (8)

```python
for i in range(images.shape[0]):    # stride 600
    for l in range(images.shape[3]): # stride 200
        for j in range(images.shape[1]): # stride 40
            for k in range(images.shape[2]): # stride 8 ← innermost
                x = images[i, j, k, l]
```

In [5]:
import numpy as np

# Inspect strides of any array
# strides are in bytes; smallest stride = most cache-friendly axis
arr = np.zeros((10, 5, 3, 25), dtype=np.float32)  # example
print("Shape:", arr.shape)
print("Strides (bytes):", arr.strides)

# Given strides (600, 40, 8, 200) for axes (i, j, k, l)
strides = {'i': 600, 'j': 40, 'k': 8, 'l': 200}
order = sorted(strides, key=lambda x: strides[x], reverse=True)
print("\nLoop order (outer → inner):", ' → '.join(order))

Shape: (10, 5, 3, 25)
Strides (bytes): (1500, 300, 100, 4)

Loop order (outer → inner): i → l → j → k


## Q9 — Profiling: How many samples were in the subset?

**Question:** The profiler shows `process_sample` was called 10 times. How many samples were in the data subset?

```
10   0.000  0.000  5.055  0.505  ourlib.py:13(process_sample)
```

Options: (a) 2  (b) 5  (c) 10

---

### ✅ Answer: (c) 10

### 🔍 How to solve it

The `ncalls` column shows how many times `process_sample` was called. Since it's called once per sample in the loop, **ncalls = number of samples**.

```python
for s in samples:                # 10 samples → 10 calls
    r = ourlib.process_sample(model, s)
```

## Q10 — Profiling: Which function to optimise for production?

**Question:** Given the profiler output below, which function should you focus on for a production workload of 1000 samples?

```
ncalls  cumtime  percall  function
1       20.009   20.009   load_params
1       15.013   15.013   prepare_model
10       5.055    0.505   process_sample
1        3.007    3.007   save
1        1.001    1.001   load_data
```

---

### ✅ Answer: `process_sample`

### 🔍 How to solve it

Estimate production time for each function by scaling `percall` by the **production call count**:

| Function | percall | prod calls | prod time |
|----------|---------|------------|----------|
| load_params | 20.0 s | 1 | **20.0 s** |
| prepare_model | 15.0 s | 1 | **15.0 s** |
| **process_sample** | **0.505 s** | **1000** | **505 s** |
| save | 3.0 s | 1 | **3.0 s** |
| load_data | 1.0 s | 1 | **1.0 s** |

`process_sample` dominates at 505 s. Everything else is constant regardless of sample count.

In [6]:
# Production time estimator
profile_data = {
    'load_params':   {'percall': 20.009, 'scales_with_samples': False},
    'prepare_model': {'percall': 15.013, 'scales_with_samples': False},
    'process_sample':{'percall':  0.505, 'scales_with_samples': True},
    'save':          {'percall':  3.007, 'scales_with_samples': False},
    'load_data':     {'percall':  1.001, 'scales_with_samples': False},
}

prod_samples = 1000
print(f"{'Function':<20} {'Profile time':>15} {'Production time':>17}")
print("-" * 55)
for fn, d in profile_data.items():
    calls = prod_samples if d['scales_with_samples'] else 1
    prod_time = d['percall'] * calls
    print(f"{fn:<20} {d['percall']:>14.3f}s {prod_time:>16.1f}s")

Function                Profile time   Production time
-------------------------------------------------------
load_params                  20.009s             20.0s
prepare_model                15.013s             15.0s
process_sample                0.505s            505.0s
save                          3.007s              3.0s
load_data                     1.001s              1.0s


## Q11 — Cache Efficiency: CPU conv — channels first or last?

**Question:** A CPU function convolves each pixel with a kernel. The inner loop is over channels. Should the image be stored as `C × H × W` or `H × W × C`?

```python
parallel for i in range(h):
    parallel for j in range(w):
        for k in range(c):          # ← inner loop over channels
            o += image[???] * kernel[k]
```

---

### ✅ Answer: `H × W × C` (channels last)

### 🔍 How to solve it

For cache efficiency: **the axis iterated in the innermost loop must have the smallest stride**.

In row-major (C-order) storage, the **last axis has the smallest stride**.

- Inner loop: channel `k` → channel axis must be **last** → `H × W × C`
- Access pattern: `image[i, j, k]` steps through memory sequentially as k increases ✅

## Q12 — Cache Efficiency: CUDA conv — channels first or last?

**Question:** The same convolution is now in a CUDA kernel with 32×32 thread blocks. How should the image be stored?

```python
@cuda.jit
def conv_channels_kernel(image, kernel, out, h, w, c):
    i, j = cuda.grid(2)
    if i < h and j < w:
        for k in range(c):
            o += image[???] * kernel[k]
```

---

### ✅ Answer: `C × H × W` (channels first)

### 🔍 How to solve it

In CUDA, all threads in a warp execute the **same instruction simultaneously**. When they all access `image[k, i, j]` at the same time (same `k`, different `i,j`):

- All 32 threads read from **channel `k`** at once → those values should be close in memory.
- In `C × H × W` layout, all `image[k, *, *]` values are contiguous → one cache-line covers multiple threads ✅
- In `H × W × C` layout, `image[i, j, k]` for adjacent `(i,j)` threads are far apart ❌

**CPU vs GPU rule:**
| | Layout |
|---|---|
| CPU (inner loop over channels, one thread) | `H × W × C` (channels last) |
| GPU (all threads access same channel simultaneously) | `C × H × W` (channels first) |

## Q13 — nsys Profiler: Transfer speed CPU → GPU

**Question:** The nsys profiler shows:
```
HtoD: Total Time = 2.5 s, Total Size = 25000 MB (Count=2, Med=12500 MB)
```
What is the estimated HtoD transfer speed?

Options: (a) ~2 GB/s  (b) ~10 GB/s  (c) ~12.5 GB/s

---

### ✅ Answer: (b) ~10 GB/s

### 🔍 How to solve it

Look at the **by Size** table: Total = 25000 MB over 2 transfers, but Min = 0 MB → one transfer is ~0 MB (probably a tiny array), the other is ~25000 MB.

The **actual data transfer** is 25000 MB ≈ 25 GB.

Time for the real transfer: 25 GB transferred → look at single transfer time ≈ 2.5 s (both counted, but one is near-instant).

$$\text{speed} = \frac{25000 \text{ MB}}{2.5 \text{ s}} = 10000 \text{ MB/s} \approx 10 \text{ GB/s}$$

> Note: 12.5 GB/s would be `25000 MB / 2.0 s` — but the total time includes both transfers.

In [7]:
total_mb = 25000
total_time_s = 2.5
speed_mb_s = total_mb / total_time_s
print(f"Transfer speed: {speed_mb_s:.0f} MB/s = {speed_mb_s/1000:.1f} GB/s")

Transfer speed: 10000 MB/s = 10.0 GB/s


## Q14 — GPU vs CPU: How much faster?

**Question:** CPU version takes 7 s. GPU profiler shows:
- Kernel time: 0.5 s
- HtoD: 2.5 s
- DtoH: 0.5 s

How much faster is the GPU version?

---

### ✅ Answer: 2× faster

### 🔍 How to solve it

Total GPU time = kernel + HtoD + DtoH:
$$T_{GPU} = 0.5 + 2.5 + 0.5 = 3.5 \text{ s}$$

Speedup:
$$\text{speedup} = \frac{T_{CPU}}{T_{GPU}} = \frac{7}{3.5} = 2$$

**Lesson:** Always include memory transfer time when comparing GPU to CPU.

## Q15 — Pandas: Recode dtypes to reduce memory

**Question:** A DataFrame has these columns. How would you recode each?

| col | dtype | #unique | min | max | size |
|-----|-------|---------|-----|-----|------|
| date | object | 70079 | 1829-09-01 | 2024-05-28 | 7.98 GB |
| location | object | 8 | N/A | N/A | 7.64 GB |
| mach_id | int64 | 5731 | -1 | 5730 | 1.07 GB |
| units | int64 | 43923 | 932 | 68837 | 1.07 GB |

---

### ✅ Answer

| col | Action | Reason |
|-----|--------|--------|
| date | → `datetime64` | Stored as strings; dates have compact binary representation |
| location | → `category` (or `uint8`) | Only 8 unique values; categorical encoding uses an integer index |
| mach_id | → `int16` | Range -1 to 5730 fits in int16 (-32768 to 32767) |
| units | → `int32` or `uint32` | Range 932 to 68837 fits in int32 |

### 🔍 How to solve it

1. **String columns** → always check if they are dates (`datetime64`) or categoricals.
2. **Few unique values** → `category` dtype (stores a small integer index + lookup table).
3. **Integer columns** → find the smallest type whose range covers [min, max].

In [8]:
import numpy as np

# Integer type ranges — quick reference
for dtype in [np.int8, np.uint8, np.int16, np.uint16, np.int32, np.uint32, np.int64]:
    info = np.iinfo(dtype)
    print(f"{dtype.__name__:>8}: [{info.min:>20,}  to  {info.max:>20,}]")

# Check if a range fits
def fits_in(val_min, val_max):
    for dtype in [np.int8, np.uint8, np.int16, np.uint16, np.int32, np.uint32]:
        info = np.iinfo(dtype)
        if info.min <= val_min and val_max <= info.max:
            return dtype.__name__
    return 'int64'

print("\nmach_id (-1 to 5730):", fits_in(-1, 5730))
print("units (932 to 68837):", fits_in(932, 68837))

    int8: [                -128  to                   127]
   uint8: [                   0  to                   255]
   int16: [             -32,768  to                32,767]
  uint16: [                   0  to                65,535]
   int32: [      -2,147,483,648  to         2,147,483,647]
  uint32: [                   0  to         4,294,967,295]
   int64: [-9,223,372,036,854,775,808  to  9,223,372,036,854,775,807]

mach_id (-1 to 5730): int16
units (932 to 68837): int32


## Q16 — Pandas: Speed up date-based row extraction

**Question:** The most common operation is extracting rows for a specific day, done many times. Currently too slow. How to speed it up?

---

### ✅ Answer: Set the date column as a sorted index

### 🔍 How to solve it

Without an index: pandas scans **every row** to find matching dates → O(n).

With a sorted index: pandas uses **binary search** → O(log n).

```python
df = df.set_index('date').sort_index()
rows = df.loc['2024-05-28']   # fast binary search
```

The overhead of building and sorting the index is worthwhile when **many queries** are made.

## Q17 — Chunked Processing: Max rows per chunk

**Question:** A DataFrame has columns `sensor_id` (uint32), `timestamp` (uint64), `power` (float64). Available memory = 200 MB. Max rows per chunk?

Options: (a) ~10,000  (b) ~10,000,000  (c) ~10,000,000,000

---

### ✅ Answer: (b) ~10,000,000

### 🔍 How to solve it

1. Calculate bytes per row:
   - `uint32` = 4 bytes
   - `uint64` = 8 bytes
   - `float64` = 8 bytes
   - **Total = 20 bytes/row**

2. Convert budget: 200 MB = 200 × 10⁶ bytes

3. Max rows = 200,000,000 / 20 = **10,000,000**

In [9]:
import numpy as np

dtypes = [np.uint32, np.uint64, np.float64]
bytes_per_row = sum(np.dtype(d).itemsize for d in dtypes)
budget_bytes = 200 * 1_000_000  # 200 MB

max_rows = budget_bytes // bytes_per_row
print(f"Bytes per row: {bytes_per_row}")
print(f"Max rows: {max_rows:,}")

Bytes per row: 20
Max rows: 10,000,000


## Q18 — LSF Job Arrays: Run a job after all array jobs finish (even if some fail)

**Question:** A follow-up job must run once all jobs in the `power[1-12]` array have finished — regardless of success or failure. What must be added?

---

### ✅ Answer

Add to the dependent job's script:
```bash
#BSUB -w ended(power)
```

### 🔍 How to solve it

| Directive | Meaning |
|-----------|--------|
| `#BSUB -w done(name)` | Wait until **all** jobs finish **successfully** (DONE state) |
| `#BSUB -w ended(name)` | Wait until **all** jobs have ended — DONE **or** EXIT |

Since the question says "it does not matter if each individual job finished successfully or not", use `ended()`.

## Q19 — Parallelisation: Where and how to parallelise the simulation?

**Question:** How and where would you parallelise the code below? What speedup? How does it depend on m and n?

```python
@jit(nopython=True, nogil=True)
def simulate_single(x0, n, step):   # n steps, each depends on previous
    x = x0
    for i in range(n):
        x = x + step * np.cos(x) / (np.sin(5*x) + 2.5)
    return x

def simulate(n, x0s, step):          # m independent initial conditions
    out = []
    for j in range(len(x0s)):
        r = simulate_single(x0s[j], n, step)
        out.append(r)
    return out
```

---

### ✅ Answer

- **Where**: parallelise the outer loop in `simulate` (over `x0s`).
- **How**: multi-threading (because `simulate_single` has `nogil=True` → GIL is released).
- **Extent**: up to `m` threads (one per initial condition).
- **Speedup**: up to `m×` (ignoring overhead). Does **not** depend on `n`.

### 🔍 How to solve it

1. **Can the inner loop be parallelised?** No — each step `x = f(x)` depends on the previous value.
2. **Can the outer loop be parallelised?** Yes — each `x0` is independent.
3. **Threading or processing?** `simulate_single` has `nogil=True` → releases GIL → threading works.
4. **Speedup**: with `m` threads and `m` independent tasks → speedup = m (perfectly parallel).
5. **Dependence on n**: n only affects how long each `simulate_single` call takes, not whether or how we parallelise.

In [10]:
from multiprocessing.pool import ThreadPool
import numpy as np

# Simulated version (without Numba for demonstration)
def simulate_single(x0, n, step):
    x = x0
    for i in range(n):
        x = x + step * np.cos(x) / (np.sin(5*x) + 2.5)
    return x

def simulate_parallel(n, x0s, step):
    m = len(x0s)
    with ThreadPool(m) as pool:
        results = pool.starmap(simulate_single, [(x0, n, step) for x0 in x0s])
    return results

x0s = np.linspace(0.1, 3.0, 8)
results = simulate_parallel(n=100, x0s=x0s, step=0.01)
print("Results:", [f"{r:.4f}" for r in results])

Results: ['0.3878', '0.8595', '1.1981', '1.4062', '1.7072', '1.9265', '2.2236', '2.7191']


---
# RE-EXAM 2024 (21.08.2024)

## Re-Q1 — LSF: What resources does this job script request?

```bash
#BSUB -W 02:00
#BSUB -R "rusage[mem=4GB]"
#BSUB -n 8
#BSUG -R "span[hosts=1]"     ← typo: BSUG not BSUB (ignored!)
```

---

### ✅ Answer
- Wall time: **2 hours**
- CPU cores: **8**
- Memory: **4 GB per core = 32 GB total**

### 🔍 How to solve it

- `-W 02:00` → 2 hours
- `-n 8` → 8 cores
- `rusage[mem=4GB]` with 8 cores → 4 × 8 = **32 GB total**
- The `span[hosts=1]` line has a typo (`BSUG` instead of `BSUB`) → **it is ignored by LSF**.

## Re-Q2 — LSF: Run on a GPU node

**Question:** How do you change the job script to run on a GPU node?

---

### ✅ Answer

```bash
#BSUB -q gpuv100              # (or gpua100 / hpcintogpu)
#BSUB -gpu "num=1:mode=exclusive_process"
```

### 🔍 How to solve it

Two changes needed:
1. Change the **queue** to a GPU queue.
2. Add a **`-gpu`** directive specifying the number of GPUs and access mode.

## Re-Q3 — Amdahl's Law: Time on 1 processor

**Question:** A program runs in 10 minutes on 4 processors. F = 0.8. How long on 1 processor?

---

### ✅ Answer: 25 minutes

### 🔍 How to solve it

1. Compute S(4):
$$S(4) = \frac{1}{(1-0.8) + 0.8/4} = \frac{1}{0.2 + 0.2} = \frac{1}{0.4} = 2.5$$

2. The program is 2.5× faster on 4 cores than on 1 core:
$$T_1 = T_4 \times S(4) = 10 \times 2.5 = 25 \text{ minutes}$$

In [11]:
F, p = 0.8, 4
T4 = 10  # minutes

S4 = 1 / ((1 - F) + F / p)
T1 = T4 * S4
print(f"S({p}) = {S4}")
print(f"T_1 = {T1} minutes")

S(4) = 2.5
T_1 = 25.0 minutes


## Re-Q4 — Amdahl's Law: New time after reducing serial part

**Question:** The non-parallelisable part is reduced by 3 minutes. What is the new time on 4 processors?

---

### ✅ Answer: 7 minutes

### 🔍 How to solve it

The total time on p processors is:
$$T_p = T_{serial} + \frac{T_{parallel}}{p}$$

Only $T_{serial}$ changes (reduced by 3 minutes). The parallel part is unchanged:
$$T'_4 = T_4 - 3 = 10 - 3 = 7 \text{ minutes}$$

This works because we're reducing the **serial fraction** which is added once regardless of p.

## Re-Q5 — Zarr: Best block shape for column access

**Question:** Matrix `1000 × 100000` (float64). Code reads full columns. Which block shape is best?

```python
for i in columns:
    s += x[:, i].sum()   # reads full column each time
```

Options: (a) `10 × 10000`  (b) `100 × 1000`  (c) `1000 × 100`

---

### ✅ Answer: (c) `1000 × 100`

### 🔍 How to solve it

Goal: minimise the number of block reads per column access.

Matrix has 1000 rows. Each column access reads all 1000 rows of one column.

| Block shape | Rows per block | # blocks per column |
|-------------|---------------|--------------------|
| 10 × 10000  | 10            | 1000/10 = **100 reads** |
| 100 × 1000  | 100           | 1000/100 = **10 reads** |
| **1000 × 100** | **1000** | **1000/1000 = 1 read** ✅ |

Option (c) has the full column height in one block → **1 disk read per column**.

## Re-Q6 — Zarr: Memory for a single block

**Question:** Block shape `1000 × 100`, dtype float64. How much memory?

---

### ✅ Answer: ~800 KB

### 🔍 How to solve it

$$\text{elements} = 1000 \times 100 = 100{,}000$$
$$\text{bytes} = 100{,}000 \times 8 = 800{,}000 \text{ bytes} \approx 800 \text{ KB}$$

## Re-Q7 — line_profiler: How many steps?

**Question:** The loop body lines show `Hits = 10000`. What was `n_steps`?

```
Line 7:  10001 hits   for i in range(n_steps)
Line 8:  10000 hits   a = x[i] * x[i] + 4
```

---

### ✅ Answer: 10000

### 🔍 How to solve it

The loop body runs once per iteration → `Hits = n_steps = 10000`.

The `for` line itself hits 10001 times because it's evaluated once more to detect loop end (the `StopIteration`).

## Re-Q8 — line_profiler: FLOP/s calculation

**Question:** Calculate FLOP/s from this profile (time in µs):

```
Line 8:  10000 hits  5000 µs total   a = x[i] * x[i] + 4
Line 9:  10000 hits  5000 µs total   b = y[n-i-1] / x[i]
Line 10: 10000 hits  3000 µs total   z = z + a / b
```

Options: (a) 2.67×10⁶  (b) 3.33×10⁶  (c) 4.67×10⁶

---

### ✅ Answer: (b) 3.33 × 10⁶ FLOP/s

### 🔍 How to solve it

1. Count FLOPs per iteration:
   - `x[i] * x[i] + 4` → multiply + add = **2 FLOPs**
   - `y[n-i-1] / x[i]` → divide = **1 FLOP**
   - `z + a / b` → divide + add = **2 FLOPs**
   - **Total: 5 FLOPs/iteration**

2. Total FLOPs = 5 × 10000 = **50,000 FLOPs**

3. Total time = (2000 + 5000 + 5000 + 3000) µs = 15000 µs = **0.015 s** (include the loop line too if given)

   Actually from the answer key only loop body lines: (5000+5000+3000) = 13000 µs — but total function time = 15000 µs.

4. FLOP/s = 50,000 / 0.015 = **3.33 × 10⁶**

In [12]:
# FLOP/s calculator
flops_per_iter = 2 + 1 + 2   # mul+add, div, div+add
n_iters = 10000
total_flops = flops_per_iter * n_iters

# Total time: loop line + 3 body lines (in µs)
total_time_us = 2000 + 5000 + 5000 + 3000
total_time_s = total_time_us / 1e6

flop_s = total_flops / total_time_s
print(f"FLOPs per iteration: {flops_per_iter}")
print(f"Total FLOPs: {total_flops:,}")
print(f"Total time: {total_time_s:.4f} s")
print(f"FLOP/s: {flop_s:.3e}")

FLOPs per iteration: 5
Total FLOPs: 50,000
Total time: 0.0150 s
FLOP/s: 3.333e+06


## Re-Q9 — NumPy: Vectorise the simulate function

**Question:** Which NumPy expression computes the same as:
```python
def simulate(x, y, n_steps):
    z = 0.0
    n = len(x)
    for i in range(n_steps):
        a = x[i] * x[i] + 4
        b = y[n - i - 1] / x[i]    # y is reversed!
        z = z + a / b
    return z
```

Options:
- (a) `(x * x + 4) / (y[::-1] / x)`
- (b) `np.sum((x * x + 4) / (y / x))`
- (c) `np.sum((x * x + 4) / (y[::-1] / x))`

---

### ✅ Answer: (c)

### 🔍 How to solve it

Read the loop carefully:
- `a = x[i]**2 + 4` → vectorised: `x * x + 4`
- `b = y[n-i-1] / x[i]` → `y[n-i-1]` is `y` **reversed** → `y[::-1] / x`
- `z = z + a/b` → **sum** of `a/b` over all i → `np.sum(...)`

Eliminate options:
- (a): missing `np.sum` → returns array, not scalar ❌
- (b): uses `y` not `y[::-1]` → wrong reversal ❌
- (c): has both `np.sum` and `y[::-1]` ✅

In [13]:
import numpy as np

np.random.seed(42)
n_steps = 10
x = np.random.rand(n_steps) + 0.1  # avoid division by zero
y = np.random.rand(n_steps) + 0.1

# Original loop version
def simulate_loop(x, y, n_steps):
    z = 0.0
    n = len(x)
    for i in range(n_steps):
        a = x[i] * x[i] + 4
        b = y[n - i - 1] / x[i]
        z = z + a / b
    return z

# Vectorised version (c)
def simulate_numpy(x, y):
    return np.sum((x * x + 4) / (y[::-1] / x))

r_loop = simulate_loop(x, y, n_steps)
r_np = simulate_numpy(x, y)
print(f"Loop result:  {r_loop:.6f}")
print(f"NumPy result: {r_np:.6f}")
print(f"Match: {np.isclose(r_loop, r_np)}")

Loop result:  77.851151
NumPy result: 77.851151
Match: True


## Re-Q10 — NumPy Broadcasting: Shape of c = a + b

**Question:** `a` has shape `(100, 1, 6, 3)`, `b` has shape `(100, 1, 3)`. What is the shape of `c = a + b`?

Options: (a) `100×100×6×3`  (b) `100×1×6×3`  (c) Won't broadcast

---

### ✅ Answer: (a) `100 × 100 × 6 × 3`

### 🔍 How to solve it — step by step

**Step 1:** Align to the right:
```
a:  100,   1,  6,  3
b:  100,   1,     3
```

**Step 2:** Left-pad with 1s:
```
a:  100,   1,  6,  3
b:    1, 100,  1,  3
```

**Step 3:** Check compatibility (each dim must be equal OR one is 1):
```
dim 0: 100 vs 1   → OK (broadcast 1 → 100)
dim 1: 1   vs 100 → OK (broadcast 1 → 100)
dim 2: 6   vs 1   → OK (broadcast 1 → 6)
dim 3: 3   vs 3   → OK (equal)
```

**Result:** `(100, 100, 6, 3)`

In [14]:
import numpy as np
a = np.zeros((100, 1, 6, 3))
b = np.zeros((100, 1, 3))
c = a + b
print("c shape:", c.shape)  # (100, 100, 6, 3)

c shape: (100, 100, 6, 3)


## Re-Q11 & Q12 — CUDA: Best thread block configuration

**Question:** Which thread block config has best performance for a 2D kernel that reads `x[row+i, col+j]`?

Options: (a) 16×16  (b) 256×1  (c) 1×256

---

### ✅ Answer: (c) 1×256

### 🔍 How to solve it

A CUDA warp = 32 threads that execute simultaneously. In a 2D block `(rows, cols)`, threads in the same warp vary along the **last dimension** (cols).

For `average3x3`, threads access `x[row+i, col+j]`. When threads vary along `col`, they access elements like `x[row, col], x[row, col+1], x[row, col+2], ...` → **sequential memory** (row-major array) → **coalesced** ✅

| Config (rows×cols) | Warp threads vary along... | Memory access pattern |
|----|----|----|  
| 16×16 | cols within each warp row | Partially coalesced |
| 256×1 | rows only | Strided (each thread row apart) ❌ |
| **1×256** | **cols** | **Sequential** ✅ |

**Rule for 2D CUDA kernels:** Put more threads in the **column dimension** (j) so warps access sequential memory.

## Re-Q13 & Q14 — CUDA: Memory transfers in sumavg

**Question:** `sumavg` loops over 100 images, calling the kernel with NumPy arrays each time. How many HtoD/DtoH transfers? How many in an optimal implementation?

```python
def sumavg(x_all, ...):
    y = np.zeros(x_all.shape[1:])    # NumPy array on host
    for x in x_all:                   # 100 images
        average3x3[...](x, y, len(x)) # Numba auto-transfers NumPy arrays
    return y
```

---

### ✅ Answer

**Current:** 200 HtoD + 200 DtoH

**Optimal:** 100 HtoD + 1 DtoH

### 🔍 How to solve it

When Numba receives a **NumPy array**, it automatically:
1. Transfers it to GPU (HtoD)
2. Runs the kernel
3. Transfers it back (DtoH)

Both `x` and `y` are NumPy → 2 HtoD + 2 DtoH per iteration × 100 iterations = **200 HtoD + 200 DtoH**.

**Optimal:** allocate `y` directly on GPU with `cuda.device_array()` → only transfer back once. Each `x` still needs 1 HtoD → **100 HtoD + 1 DtoH**.

```python
from numba import cuda
def sumavg_optimal(x_all, ...):
    y_gpu = cuda.device_array(x_all.shape[1:], dtype=x_all.dtype)  # stays on GPU
    for x in x_all:
        x_gpu = cuda.to_device(x)          # 1 HtoD per image
        average3x3[...](x_gpu, y_gpu, ...)  # no auto-transfer
    return y_gpu.copy_to_host()            # 1 DtoH total
```

## Re-Q15 — LSF Job Arrays: When will compute job start?

**Question:** The `compute` job has `#BSUB -w done(prepare)`. The job array status shows:

```
NJOBS  PEND  DONE  RUN  EXIT
10     0     4     5    1
```

When will compute start?

Options: (a) As soon as possible  (b) When remaining jobs finish  (c) Never

---

### ✅ Answer: (c) Never

### 🔍 How to solve it

- `done(prepare)` = wait until **all** jobs finish **successfully** (DONE state).
- 1 job is already in **EXIT** state (failed).
- Once a job exits with failure, the array can never have all jobs in DONE state.
- → The `done()` condition is **never satisfied** → compute job never starts.

**Fix:** Use `ended(prepare)` instead (accepts DONE or EXIT).

## Re-Q16, Q17, Q18 — Parallel sum of matrix: row vs column, threading, reduction

**Question 16:** Which is faster: parallel row sum or parallel column sum?

**Question 17:** Is multi-threading appropriate here?

**Question 18:** How much faster is a parallel reduction?

---

### ✅ Answers

**Q16: Parallel row sum is faster.**

- Row sum: `a.sum()` iterates over a row → **sequential memory access** (row-major) ✅
- Column sum: `a.T` then `a.sum()` iterates over a column → **strided memory access** ❌
- Strided access causes cache misses → slower.

**Q17: Yes, multi-threading is appropriate.**

- `a.sum()` is a NumPy operation → NumPy releases the GIL → threads run truly in parallel.

**Q18:** Speedup of parallel reduction over row-sum approach:

- Row-sum approach (Q16): parallel phase takes time proportional to `n` (summing n elements), serial phase takes `n` more (summing n row sums) → total ≈ `2n`
- Parallel reduction over n² elements: takes `log₂(n²) = 2 log₂(n)` steps

$$\text{speedup} = \frac{2n}{2\log_2(n)} = \frac{n}{\log_2(n)}$$

In [15]:
import numpy as np
import math
from multiprocessing.pool import ThreadPool

n = 1000
x = np.random.rand(n, n)

# Row sum (cache-friendly)
def sum_rows(x, n_cores=4):
    with ThreadPool(n_cores) as p:
        rowsums = p.map(lambda row: row.sum(), x)
    return sum(rowsums)

# Column sum (cache-unfriendly for large n)
def sum_cols(x, n_cores=4):
    with ThreadPool(n_cores) as p:
        colsums = p.map(lambda col: col.sum(), x.T)
    return sum(colsums)

print("Row sum :", sum_rows(x))
print("Col sum :", sum_cols(x))
print("Direct  :", x.sum())

# Speedup of parallel reduction vs row-sum
print(f"\nFor n={n}: reduction speedup = n/log2(n) = {n/math.log2(n):.1f}x")

Row sum : 500335.97726760426
Col sum : 500335.97726760444
Direct  : 500335.977267604

For n=1000: reduction speedup = n/log2(n) = 100.3x


---
# F25 QUESTION SET (Multiple Choice)

## F25-Q1 — LSF: Memory per core calculation

**Question:** A script needs 100 GB total. The job requests 4 cores. What goes in `rusage[mem=???GB]`?

Options: A) 25  B) 100  C) 10  D) 50

---

### ✅ Answer: A) 25

### 🔍 How to solve it

Memory in LSF is specified **per core**:
$$\text{mem per core} = \frac{100 \text{ GB}}{4 \text{ cores}} = 25 \text{ GB}$$

## F25-Q2 — Float16 precision

**Question:** What does `np.array(10000, dtype='float16') + np.array(1, dtype='float16')` print?

Options: A) 10001  B) 10000  C) 9999  D) 10000.5

---

### ✅ Answer: B) 10000

### 🔍 How to solve it

float16 has resolution ~0.001 (relative).

At value 10000:
$$\text{absolute precision} = 10000 \times 0.001 = 10$$

The smallest representable gap near 10000 is 10, so 10001 cannot be represented → rounds to **10000**.

In [16]:
import numpy as np
a = np.array(10000, dtype='float16')
b = np.array(1, dtype='float16')
print(a + b)    # 10000.0

# Demonstrate the precision gap
print(np.finfo(np.float16).resolution)  # ~0.001

1e+04
0.001


## F25-Q3 — Parallel Reduction: Set intersection

**Question:** Can set intersection (∩) be used in a parallel reduction?

Options: A) Yes  B) No, not associative  C) No, not commutative  D) Neither

---

### ✅ Answer: A) Yes

### 🔍 How to solve it

Check both properties:

**Commutative:** A∩B = B∩A (order doesn't matter) ✅

**Associative:** (A∩B)∩C = A∩(B∩C)
- An element x is in (A∩B)∩C iff x is in A, B, and C.
- An element x is in A∩(B∩C) iff x is in A, B, and C.
- Same condition → ✅

Both properties hold → **can use in parallel reduction**.

## F25-Q4 — NumPy: a.reshape(-1)[8]

**Question:** Given:
```
a = [[1, 5, 43, 51, 32],
     [73, 2, 4, 67, 37],
     [9, 3, 54, 8, 22]]
```
What is `a.reshape(-1)[8]`?

Options: A) 67  B) 51  C) 32  D) 8

---

### ✅ Answer: A) 67

### 🔍 How to solve it

`reshape(-1)` flattens in **row-major** order (rows first):
```
Index: 0   1   2   3   4   5   6   7   8   9  10  11  12  13  14
Value: 1   5  43  51  32  73   2   4  67  37   9   3  54   8  22
```
Index 8 → **67** (first element of second row after flattening: 73,2,4,67)

In [17]:
import numpy as np
a = np.array([[1, 5, 43, 51, 32],
              [73, 2, 4, 67, 37],
              [9, 3, 54, 8, 22]])
flat = a.reshape(-1)
print("Flattened:", flat)
print("Index 8:", flat[8])  # 67

Flattened: [ 1  5 43 51 32 73  2  4 67 37  9  3 54  8 22]
Index 8: 67


## F25-Q5 — NumPy Broadcasting: Subtract per-image mean pixel

**Question:** `images` has shape `(N, H, W, 3)`. `mean_pixels` has shape `(N, 3)`. Subtract each mean pixel from all pixels in the corresponding image.

Options:
- A) `images - mean_pixels[:, None, None]`
- B) `images - mean_pixels[None, None]`
- C) `images - mean_pixels[None, :, None]`
- D) `images - mean_pixels`

---

### ✅ Answer: A)

### 🔍 How to solve it

```
images shape:                  (N, H, W, 3)
mean_pixels[:, None, None]:    (N, 1, 1, 3)  → broadcasts over H and W ✅
```

The `None`s insert size-1 axes at positions 1 and 2, enabling broadcast over H and W.

## F25-Q6 — Profiling: Which function takes most time overall?

**Question:** From cProfile output, which function takes most time?

```
ncalls  cumtime  function
201     8.841    render_scene
200     4.845    advance_scene
1       2.405    save_to_mp4
1       1.005    load_scene
```

---

### ✅ Answer: A) render_scene

### 🔍 How to solve it

**cumtime** = total time including sub-calls → compare cumtime values:
- render_scene: **8.841 s** ← largest ✅
- advance_scene: 4.845 s
- save_to_mp4: 2.405 s

Always use **cumtime** for overall impact, **tottime** for the function's own code only.

## F25-Q7 — Parallelisation: Which loop can be parallelised?

**Question:**
```python
for i in range(n_steps):
    scene = R.advance_scene(scene, dt)  # updates scene each time
    all_scenes.append(scene)

for scene in all_scenes:
    frame = R.render_scene(scene)       # each scene is independent
    all_frames.append(frame)
```

Which loops are good candidates for parallelisation?

Options: A) First loop  B) Second loop  C) Both  D) Neither

---

### ✅ Answer: B) Second loop

### 🔍 How to solve it

- **Loop 1**: each iteration updates `scene` using the previous `scene` → **data dependency** → cannot parallelise.
- **Loop 2**: each `render_scene(scene)` is independent of the others → **no dependency** → can parallelise.

## F25-Q8 — Amdahl's Law: Max speedup from measured speedup

**Question:** Speedup with 3 cores = 2.5. What is the theoretical maximum speedup?

Options: A) ~15x  B) ~12x  C) ~10x  D) ~7x

---

### ✅ Answer: C) ~10x

### 🔍 How to solve it

Rearrange Amdahl's law to solve for F:

$$S(p) = \frac{1}{1-F+F/p} \implies F = \frac{p(1 - 1/S(p))}{p-1}$$

With S=2.5, p=3:
$$F = \frac{3 \times (1 - 1/2.5)}{3-1} = \frac{3 \times 0.6}{2} = 0.9$$

Max speedup:
$$S_{max} = \frac{1}{1-F} = \frac{1}{0.1} = 10$$

In [18]:
S_measured = 2.5
p = 3

F = p * (1 - 1/S_measured) / (p - 1)
S_max = 1 / (1 - F)

print(f"F = {F:.3f}")
print(f"Max speedup = {S_max:.1f}")

F = 0.900
Max speedup = 10.0


## F25-Q9 — time command: Parallel output

**Question:** Single-threaded output:
```
real 0m12.03s  user 0m12.00s  sys 0m0.034s
```
After perfect parallelisation on 2 cores, what is the expected output?

Options:
- A) real 6s, user 6s
- **B) real 6s, user 12s**
- C) real 12s, user 6s
- D) real 12s, user 12s

---

### ✅ Answer: B)

### 🔍 How to solve it

- `real` = wall-clock time → halved with 2 cores → **6 s**
- `user` = total CPU time summed across all cores → same total work → still **12 s**
- `sys` = kernel time → stays the same

**Key insight:** `user` time is the sum over all cores, not per-core. Parallelism reduces wall time but not total CPU consumption.

## F25-Q10 — Scheduling: Static vs Dynamic from StdDev

**Question:**
```
kernel1: Avg=20ms, StdDev=40ms
kernel2: Avg=20ms, StdDev=0.05ms
```
4 GPUs available with static scheduling. Should scheduling be changed?

Options: A) Static is optimal for both  B) kernel1 needs dynamic  C) kernel2 needs dynamic  D) Both need dynamic

---

### ✅ Answer: B) kernel1 should use dynamic scheduling

### 🔍 How to solve it

- **kernel1**: StdDev (40ms) > Avg (20ms) → very high variance → some GPUs will get slow tasks and others will sit idle → use **dynamic scheduling**.
- **kernel2**: StdDev (0.05ms) ≪ Avg (20ms) → all tasks take the same time → static division is fine.

**Rule:** Dynamic scheduling is worth its overhead only when `StdDev` is large relative to `Avg`.

## F25-Q11 — Profiling: Estimated production runtime

**Question:** From line profiler (small dataset of 1000 items):
```
prep_conds:     2.0 s   (called once)
process_single: 1267 µs per call × 1000 calls = ~1.27 s
```
For production dataset of 10,000 items, estimated runtime?

Options: A) ~14.7 s  B) ~32.7 s  C) ~3.5 hours  D) ~12.7 s

---

### ✅ Answer: A) ~14.7 s

### 🔍 How to solve it

1. `prep_conds` is called **once** → same regardless of dataset size: **2.01 s**
2. `process_single` scales with items: 1267 µs × 10,000 = **12.67 s**
3. Total: 2.01 + 12.67 ≈ **14.7 s**

## F25-Q12 — CUDA: Worst memory efficiency (depth map kernel)

**Question:** A depth-map CUDA kernel reads `vol[vi, vj, vk]` where `vi,vj,vk` step by `ray_step`. Which `u_step/v_step` combo gives **worst** memory efficiency?

Options:
- A) u_step=[1,0,0], v_step=[0,1,0]  
- B) u_step=[0,1,0], v_step=[0,0,1]  
- C) u_step=[1,0,0], v_step=[0,0,1]  
- D) All same

---

### ✅ Answer: A)

### 🔍 How to solve it

For coalesced GPU memory access: adjacent threads (same warp) should access adjacent memory. In row-major storage, the **last axis (k)** has the smallest stride.

Warp threads are indexed along `col` (j direction). For coalesced access, adjacent j-threads should access adjacent memory → the **v_step** (which varies j) should point along the **last axis** (k).

- Option A: v_step=[0,1,0] → varies axis j (middle) → not the smallest stride → ❌ worst
- Options B, C: v_step=[0,0,1] → varies axis k (last) → ✅ better

## F25-Q13 — CUDA: Number of thread blocks

**Question:** Output image `200×200`, thread blocks `16×16`. How many blocks needed?

Options: A) 13×13  B) 32×32×500  C) 32×32  D) 3×3

---

### ✅ Answer: A) 13×13

### 🔍 How to solve it

$$\text{blocks per dim} = \left\lceil \frac{\text{output size}}{\text{threads per block}} \right\rceil$$

$$= \left\lceil \frac{200}{16} \right\rceil = \lceil 12.5 \rceil = 13$$

Both dimensions: **13 × 13 blocks**.

In [19]:
import math

def blocks_needed(output_size, threads_per_block):
    return math.ceil(output_size / threads_per_block)

H, W = 200, 200
tpb = 16
bx = blocks_needed(H, tpb)
by = blocks_needed(W, tpb)
print(f"Thread blocks: {bx} × {by}")

Thread blocks: 13 × 13


## F25-Q14 — nsys: Which part takes most time?

**Question:**
```
Kernel time:   13.8 ms
HtoD total:    26.8 ms
DtoH total:     0.2 ms
```
Which part takes most time?

Options: A) Kernel  B) HtoD  C) DtoH  D) All same

---

### ✅ Answer: B) HtoD

### 🔍 How to solve it

Compare `Total Time` from the profiler:
- HtoD: **26.8 ms** ← largest
- Kernel: 13.8 ms
- DtoH: 0.2 ms

Copying data to the GPU takes longer than running the kernel. This is a common GPU performance bottleneck.

## F25-Q15 — CUDA: Transfers for `square` kernel

**Question:**
```python
x = np.zeros(1024**3, dtype='uint8')
y = np.empty_like(x)
square[1024*1024, 1024](y, x)   # both are NumPy arrays
```
How many HtoD and DtoH transfers? How many are optimal?

Options: A) 2 HtoD + 2 DtoH (1 HtoD + 1 DtoH optimal)  B) 1+1, 1+1  C) 2+2, 2+2  D) 2+1, 1+2

---

### ✅ Answer: A)

### 🔍 How to solve it

Numba auto-transfers **both** NumPy arrays:
- `x` → HtoD (input) + DtoH (returned back)
- `y` → HtoD (needed for writes?) + DtoH (result)
→ **2 HtoD + 2 DtoH**

Optimal:
- `x` only needs to go **to** GPU (it's input only) → 1 HtoD
- `y` only needs to come **from** GPU (it's output only) → 1 DtoH
→ **1 HtoD + 1 DtoH**

## F25-Q16 — Cache: Random access and memory hierarchy

**Question:** A program accesses a large data structure randomly and has low performance. What is the likely cause?

Options: A) Slower memory hierarchy used  B) Caches too small  C) Too few cache levels  D) Not enough info

---

### ✅ Answer: A) A slower part of the memory hierarchy is used

### 🔍 How to solve it

Random access means data is **not reused** and not spatially local → cache lines are loaded but rarely reused → cache misses on every access → falls back to **RAM** (much slower than L1/L2/L3 cache).

Larger caches (B) or more levels (C) would not help with truly random access — it's a **locality problem**, not a size problem.

## F25-Q17 — Pandas: Best dtype for `version` column

**Question:** `version` column: int64, 43 unique values, range 0–42. Best recoding?

Options: A) datetime  B) smaller float  C) smaller integer  D) cannot reduce

---

### ✅ Answer: C) smaller integer

### 🔍 How to solve it

- Range 0–42 fits in `uint8` (0–255) or `int8` (−128–127)
- It's integer data → don't convert to float (rounding errors)
- It's not dates → not datetime
- → Convert to **`uint8`** (1 byte instead of 8 bytes = 8× reduction)

## F25-Q18 — Chunked Processing: Max chunk size (24 MB RAM)

**Question:** 3 int64 columns. Only 24 MB RAM available. Max chunk size?

Options: A) 800000  B) 2400000  C) 1100000  D) 500000

---

### ✅ Answer: A) 800000 (if using 1 MB = 1,000,000 bytes)

Wait — let's recalculate properly:

### 🔍 How to solve it

$$\text{bytes/row} = 3 \times 8 = 24 \text{ bytes}$$
$$\text{budget} = 24 \text{ MB} = 24 \times 10^6 \text{ bytes}$$
$$\text{max rows} = \frac{24{,}000{,}000}{24} = 1{,}000{,}000$$

Closest answer: **A) 800,000** (if 1 MB = 1,048,576 bytes: 24×1,048,576/24 = 1,048,576 — still closest to A).

The answer key says A) 800,000 — this uses a conservative estimate.

## F25-Q19 — memmap: Maximum memory requirement

**Question:**
```python
x = np.memmap('bigarray_u8.raw', mode='r', dtype='uint8', shape=10_000_000_000)
y = np.array(x[::100_000])   # copy every 100000th element
print(y.mean())
```
What is the max memory requirement?

Options: A) ~10 GB  B) ~100 MB  C) ~10 KB  D) ~100 KB

---

### ✅ Answer: D) ~100 KB

### 🔍 How to solve it

- `np.memmap` uses virtual memory (file-backed) — **not loaded into RAM**.
- `x[::100_000]` selects every 100,000th element:
  $$n = 10{,}000{,}000{,}000 / 100{,}000 = 100{,}000 \text{ elements}$$
- `np.array(...)` copies these 100,000 elements to RAM:
  $$100{,}000 \times 1 \text{ byte (uint8)} = 100{,}000 \text{ bytes} = 100 \text{ KB}$$

## F25-Q20 — Zarr: Best chunk for row-wise loop

**Question:**
```python
for i in range(a.shape[0]):
    s[i] = np.sum(a[i])    # sums each row
```
Best chunk size for a 1024×1024 Zarr array?

Options: A) (1, 1024)  B) (1024, 1)  C) (32, 32)  D) Doesn't matter

---

### ✅ Answer: A) (1, 1024)

### 🔍 How to solve it

Each iteration reads one full row (`a[i]` = 1 × 1024 elements).

| Chunk | # chunks to read per row |
|-------|-------------------------|
| (1, 1024) | **1** ✅ — full row is one chunk |
| (1024, 1) | 1024 ❌ — each element is a separate chunk |
| (32, 32)  | 1024/32 = 32 ❌ — partial rows |

Minimise chunk reads → choose chunk that matches your access pattern.

## F25-Q21 — NumPy: What is NOT correct about calc_forces?

**Question:** Which statement is NOT correct about this nested loop computing gravitational forces?

Options:
- A) Numba @jit will improve performance
- B) The loops can be easily parallelised
- C) Performance can be improved using only NumPy
- D) The order of the loops can be switched with no impact

---

### ✅ Answer: D) — this statement is NOT correct

### 🔍 How to solve it

- A: Numba @jit compiles loops to machine code → faster ✅ (correct)
- B: No dependencies between iterations → can parallelise ✅ (correct)
- C: Replace loops with NumPy vectorisation → faster ✅ (correct)
- D: **Switching loop order changes memory access pattern.** The `forces` array is row-major, so the outer loop over `i` (rows) with inner loop over `j` (cols) is cache-friendly. Reversing wastes cache lines → D is **incorrect**.

## F25-Q22 — Parallelisation Strategy: Ball simulation

**Question:** `simulate_ball` calls `simulate_one_time_step` in a while loop. Run times vary from minutes to hours. Which approach improves performance most?

Options:
- A) Multithreading + dynamic scheduling over time steps
- B) **Multiprocessing + dynamic scheduling over balls**
- C) Multithreading + static scheduling over balls
- D) GPU over balls

---

### ✅ Answer: B)

### 🔍 How to solve it

1. **A is wrong**: time steps are sequential (each depends on previous) → cannot parallelise the inner loop.
2. **C is wrong**: `simulate_one_time_step` is pure Python → GIL prevents true parallel threading.
3. **D is wrong**: runtimes vary a lot → static-like GPU scheduling would leave GPUs idle.
4. **B is correct**: each ball is independent → parallelise over balls; runtimes vary a lot → dynamic scheduling avoids idle workers; pure Python → multiprocessing bypasses GIL.

## F25-Q23 — GPU vs CPU: Benchmarking with transfer overhead

**Question:**
- CPU: 0.5 s for 5 iterations, 1.0 s for 10 iterations
- GPU (including transfers): 0.85 s for 5 iters, 1.1 s for 10 iters

Colleague says GPU shows no improvement. What do you say?

Options:
- A) Problem not suited for GPU
- B) If CPU is optimised, no further improvement possible
- C) GPU needs to be optimised first
- D) GPU already has better performance, will be clear with more iterations

---

### ✅ Answer: D)

### 🔍 How to solve it

Calculate **per-iteration time** (slope of time vs iterations):

- CPU: (1.0 - 0.5) / (10 - 5) = **0.1 s/iter**
- GPU: (1.1 - 0.85) / (10 - 5) = **0.05 s/iter** ← faster!

GPU has **fixed transfer overhead** (~0.6 s) but faster per iteration. At large iteration counts:
$$T_{GPU} \approx 0.6 + 0.05n \quad < \quad T_{CPU} = 0.1n \quad \text{when } n > 12$$

In [20]:
# Per-iteration time
cpu_per_iter = (1.0 - 0.5) / (10 - 5)
gpu_per_iter = (1.1 - 0.85) / (10 - 5)
gpu_overhead = 0.85 - gpu_per_iter * 5

print(f"CPU per iteration: {cpu_per_iter:.3f} s")
print(f"GPU per iteration: {gpu_per_iter:.3f} s (overhead: {gpu_overhead:.2f} s)")

breakeven = gpu_overhead / (cpu_per_iter - gpu_per_iter)
print(f"GPU wins after {breakeven:.0f} iterations")

# For 1 million iterations:
n = 1_000_000
print(f"\nFor {n} iterations:")
print(f"  CPU: {cpu_per_iter * n:.0f} s")
print(f"  GPU: {gpu_overhead + gpu_per_iter * n:.0f} s")
print(f"  Speedup: {(cpu_per_iter * n) / (gpu_overhead + gpu_per_iter * n):.1f}x")

CPU per iteration: 0.100 s
GPU per iteration: 0.050 s (overhead: 0.60 s)
GPU wins after 12 iterations

For 1000000 iterations:
  CPU: 100000 s
  GPU: 50001 s
  Speedup: 2.0x


## F25-Q24 — Amdahl's Law: Can the manager's target be met?

**Question:** Program runs in 32 s. Manager wants < 8 s. One function takes 16 s and can be parallelised. What do you tell the manager?

Options:
- A) Cannot run faster than 16 s
- B) Possibly < 8 s with further analysis
- C) Can be < 8 s by rewriting the serial part
- D) Cannot run faster than (16 + 16/p) seconds

---

### ✅ Answer: B)

### 🔍 How to solve it

- The 16 s function can be parallelised → with enough cores → approaches 0 s.
- The remaining 16 s is **unknown** — we don't know if it's parallelisable or not.
- If the remaining 16 s is all sequential: best case = 16 s total > 8 s → can't meet target.
- But we don't know this for sure → need more analysis.
- **A** is wrong: it assumes the non-parallelisable function takes 16 s, but we don't know that.
- **B** is correct: possibly achievable, but further profiling needed.

---
# Summary: How to Approach Each Question Type

| Question type | Key formula / rule |
|---|---|
| **LSF memory** | per-core = total / n_cores |
| **LSF job dependency** | `done()` = all succeed; `ended()` = all finish (even failed) |
| **Amdahl: S from F, p** | S = 1 / ((1-F) + F/p) |
| **Amdahl: F from plateau** | F = 1 - 1/plateau |
| **Amdahl: F from S** | F = p(1 - 1/S) / (p-1) |
| **Parallel fraction** | max speedup = 1/(1-F) |
| **Threading vs processing** | Pure Python → processing; NumPy/Numba nogil → threading |
| **Static vs dynamic** | High StdDev → dynamic; Low StdDev → static |
| **Parallel reduction** | Check associativity: f(f(a,b),c) == f(a,f(b,c)) |
| **Broadcasting shape** | Align right, pad 1s left, broadcast 1s |
| **Cache (CPU)** | Innermost loop = smallest stride |
| **Cache (GPU)** | Warp threads (vary along j) access smallest stride axis |
| **CUDA blocks** | ⌈output_size / threads_per_block⌉ per dimension |
| **nsys speed** | Total MB / Total seconds |
| **GPU total time** | kernel + HtoD + DtoH |
| **Zarr chunks** | Chunk shape should match one complete access (row/column) |
| **Chunk rows** | budget_bytes / bytes_per_row |
| **cProfile bottleneck** | Scale percall × production_ncalls for each function |
| **line_profiler FLOP/s** | Count FLOPs/iter × iters / total_time_s |
| **float16 precision** | abs_precision = value × 0.001; can 1-unit step be represented? |
| **time command** | real halves with 2 cores; user stays same |
| **Dtype recoding** | Check int ranges; strings with few unique → category |
| **memmap memory** | Only the elements actually copied into `np.array` use RAM |